# Algoritmos de Emparejamiento de Trayectorias

## Introducción

En este notebook, exploraremos dos algoritmos para el emparejamiento de trayectorias: el algoritmo **TrajectoryMatch (TM)** y el algoritmo **Similitud Ponderada Espacial (SWS, por sus siglas en inglés)**. Estos algoritmos se utilizan para comparar las trayectorias de objetos en movimiento basándose en sus características espaciales y temporales. 

El análisis de trayectorias tiene aplicaciones en diversas áreas, como el seguimiento de vehículos, el análisis de rutas en sistemas de navegación y el estudio del comportamiento en entornos geoespaciales. Cada algoritmo tiene un enfoque único y una justificación específica para su uso, que será detallada en las siguientes secciones.

## Algoritmo TrajectoryMatch (TM)

El algoritmo **TrajectoryMatch** fue obtenido del artículo *"Review On Trajectory Similarity Measures"* y compara dos trayectorias en función de sus relaciones direccionales y topológicas. Este enfoque combina aspectos geométricos y espaciales para analizar el alineamiento y la proximidad entre dos trayectorias.

El artículo proporciona un marco teórico que respalda la importancia de analizar tanto las relaciones direccionales como topológicas para comprender las similitudes entre trayectorias. Esta metodología es especialmente útil en escenarios donde el comportamiento direccional de los objetos es relevante, como en sistemas de tráfico o patrones migratorios.

### Función de Relación Direccional

Esta función evalúa la relación direccional entre dos puntos. Se basa en la dirección del vector que conecta ambos puntos y clasifica el movimiento según un conjunto de direcciones definidas (por ejemplo, norte, sur, este, oeste, y combinaciones intermedias).

#### Propiedades:
- **Entrada:** Dos puntos, cada uno con coordenadas (x, y).
- **Salida:** Una etiqueta que indica la dirección (por ejemplo, "noreste" o "suroeste").
- **Fórmula:** 
  ```python
  dirección = atan2(y2 - y1, x2 - x1)


# Algoritmo Similitud Ponderada Espacial (SWS)

El algoritmo **Similitud Ponderada Espacial (SWS)** se obtuvo del repositorio público de GitHub en [Gooong/TrajectoryMatching](https://github.com/Gooong/TrajectoryMatching). A diferencia de TM, este algoritmo incorpora un sistema de pesos espaciales y temporales, proporcionando un enfoque más flexible y adaptable para analizar trayectorias en diferentes contextos.

## Características del Algoritmo

- **Pesos Espaciales**: Se asignan mayores pesos a regiones consideradas más importantes para el análisis. Por ejemplo, las áreas con alta densidad de puntos pueden recibir mayor relevancia.

- **Respetuoso del Tiempo**: Incluye un componente temporal para analizar cómo evolucionan las trayectorias a lo largo del tiempo, lo que lo hace útil en aplicaciones dinámicas.

## Fórmula General:

### SWS(T1, T2):

\[
SWS(T1, T2) = \sum_{i=1}^{n} w_i \cdot d(p_{1,i}, p_{2,i})
\]

Donde:

- \( T1, T2 \): Las trayectorias a comparar.
- \( w_i \): El peso asignado al punto \( i \), que depende de la relevancia espacial.
- \( d(p_{1,i}, p_{2,i}) \): La distancia entre los puntos \( p_{1,i} \) y \( p_{2,i} \) correspondientes en las trayectorias.


In [6]:
import numpy as np
import math

# SWS: Calculate the distance between two points using the Haversine formula
def haversine(lon1, lat1, lon2, lat2):
    lon1 = math.radians(lon1)
    lat1 = math.radians(lat1)
    lon2 = math.radians(lon2)
    lat2 = math.radians(lat2)
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(min(1, a)))  # Clamp 'a' to avoid math domain error
    r = 6371  # Radius of Earth in kilometers
    return c * r * 1000  # Convert to meters

# SWS: Proximity Similarity Function (PSF) based on distance
def psf(distance):
    if distance > 3500:
        return 0
    elif distance <= 1500:
        return 1
    else:
        return 1 - (distance - 1500) / 2000

# SWS: Space Weighted Similarity (SWS) algorithm for spatial similarity
def sws(record1, record2):
    """
    Space Weighted Similarity (SWS) focusing only on spatial similarity.
    :param record1: [[x1,x2,...],[y1,y2,...]]
    :param record2: [[x1,x2,...],[y1,y2,...]]
    :return: the space weighted similarity score
    """
    assert len(record1[0]) > 1 and len(record2[0]) > 1

    xl1, yl1 = record1
    xl2, yl2 = record2

    # Calculate distances and similarity scores
    distance_list = list(map(haversine, xl1, yl1, xl2, yl2))
    similarity_list = list(map(psf, distance_list))

    # Calculate the final similarity score
    score = sum(similarity_list) / len(similarity_list)
    return score, distance_list  # Return distances to show the comparison

# Define Trajectories for testing: Equal, Slightly Similar, and Different

# Equal trajectories (no change)
trajectory_equal_1 = [[-123.3656, -123.3657, -123.3658], [48.4284, 48.4285, 48.4286]]
trajectory_equal_2 = [[-123.3656, -123.3657, -123.3658], [48.4284, 48.4285, 48.4286]]

# Slightly similar trajectories (small change in latitude and longitude)
trajectory_similar_1 = [[-123.3656, -123.3657, -123.3658], [48.4284, 48.4285, 48.4286]]
trajectory_similar_2 = [[-123.3655, -123.3656, -123.3657], [48.4285, 48.4286, 48.4287]]

# Different trajectories (more noticeable change in lat/lon)
trajectory_different_1 = [[-123.3656, -123.3657, -123.3658], [48.4284, 48.4285, 48.4286]]
trajectory_different_2 = [[-122.3656, -122.3657, -122.3658], [47.4284, 47.4285, 47.4286]]

# Calculate SWS scores for equal, similar, and different trajectories
score_equal_sws, distances_equal = sws(trajectory_equal_1, trajectory_equal_2)
score_similar_sws, distances_similar = sws(trajectory_similar_1, trajectory_similar_2)
score_different_sws, distances_different = sws(trajectory_different_1, trajectory_different_2)

# Print the results for SWS
print(f"SWS Similarity Score for Equal Trajectories: {score_equal_sws}")
print(f"Distances for Equal Trajectories: {distances_equal}")
print(f"SWS Similarity Score for Slightly Similar Trajectories: {score_similar_sws}")
print(f"Distances for Slightly Similar Trajectories: {distances_similar}")
print(f"SWS Similarity Score for Different Trajectories: {score_different_sws}")
print(f"Distances for Different Trajectories: {distances_different}")



SWS Similarity Score for Equal Trajectories: 1.0
Distances for Equal Trajectories: [0.0, 0.0, 0.0]
SWS Similarity Score for Slightly Similar Trajectories: 1.0
Distances for Slightly Similar Trajectories: [13.344804186217269, 13.344796160208727, 13.344788131533324]
SWS Similarity Score for Different Trajectories: 0.0
Distances for Different Trajectories: [133846.50682500002, 133846.42662796477, 133846.34643090965]
